In [1]:
#check cuda
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print('GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print('CPU')


GPU: Tesla P100-PCIE-16GB


# 1. Data Generation and Preparation

In [2]:
import numpy as np
#size of the matrix squared to discover a multiplication algorithm for
size = 2#3
#number of matrices to create
num_of_matrices = 10000 

#use a seed for reproducability (not to create a different dataset in every training)
##This step can be replaced by generating data once and saving it
np.random.seed(42) 
A_matrices = torch.tensor(np.random.uniform(-1, 1, size=(num_of_matrices, size, size)), dtype=torch.float32)
A_matrices = torch.round(A_matrices * 100) / 100

np.random.seed(43) 
B_matrices = torch.tensor(np.random.uniform(-1, 1, size=(num_of_matrices, size, size)),dtype=torch.float32)
B_matrices = torch.round(B_matrices * 100) / 100

#the value of matices begtween -1 and 1 to be normlized automatically

#C is the multiplication result of A and B
C_matrices = torch.tensor(np.zeros((num_of_matrices, size, size)), dtype=torch.float32)
for i in range (num_of_matrices):
    C_matrices[i] = A_matrices[i] @ B_matrices[i]

In [4]:
#flattening the matrices
A_matrices = A_matrices.reshape(num_of_matrices,-1)
B_matrices = B_matrices.reshape(num_of_matrices,-1)
C_matrices = C_matrices.reshape(num_of_matrices,-1)

A_matrices.shape

torch.Size([10000, 4])

In [5]:
# Splitting (80% for train and 20 for test) 
A_train  = A_matrices[:8000]
B_train = B_matrices[:8000]
C_train = C_matrices[:8000]

A_test = A_matrices[8000:]
B_test = B_matrices[8000:]
C_test = C_matrices[8000:]

In [6]:
#prepare data to be fed to the network

from torch.utils.data import DataLoader, TensorDataset
train_dataset = TensorDataset(A_train, B_train, C_train)
train_loader = DataLoader(train_dataset, batch_size=8000, shuffle=True)


test_dataset = TensorDataset(A_test, B_test , C_test)
test_loader = DataLoader(test_dataset, batch_size=2000, shuffle=True)

# 2. Neural network definition

In [7]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


In [8]:
#Define dot product for activation function of the hidden layer
class MultiplyActivation(nn.Module):
    def __init__(self):
        super(MultiplyActivation, self).__init__()

    def forward(self, input_set1, input_set2):
        # Element-wise multiplication of the two sets of neurons
        result = input_set1 * input_set2

        return result


In [9]:

input_mat = size*size #size of input matrices
mul = 7  # number of multiplication to be in the algorithm

# Define the neural network architecture
class SimpleANN(nn.Module):
    def __init__(self):
        super(SimpleANN, self).__init__()
        self.input_a = nn.Linear(input_mat, mul,bias=False)  # Input layer to hidden layer
        self.input_b = nn.Linear(input_mat, mul,bias=False)  # Input layer to hidden layer

        self.output_c = nn.Linear(mul, input_mat,bias=False)  # Hidden layer to output layer

        self.multiply_activation = MultiplyActivation()

        #self.input_a.weight.data.uniform_(-2, 2)
        #self.input_b.weight.data.uniform_(-2, 2)
        #self.output_c.weight.data.uniform_(-2, 2)

        # Initialize weights using Xavier initialization
        #torch.manual_seed(36)
        nn.init.xavier_uniform_(self.input_a.weight) 
        nn.init.xavier_uniform_(self.input_b.weight)
        nn.init.xavier_uniform_(self.output_c.weight)

        

    def forward(self, a,b):

        a,b = self.input_a(a), self.input_b(b)

        #x = a*b
        x = self.multiply_activation(a,b)

        x = self.output_c(x)

        return x   


In [10]:
# Instantiate the model
model = SimpleANN()
model.to(device)

#Loss function
#criterion = nn.L1Loss()
criterion = nn.MSELoss()
criterion.to(device)

# Define an optimizer
#optimizer = optim.AdamW(model.parameters(), lr=1e-3)
optimizer = optim.Adam(model.parameters(), lr=1e-3)



In [11]:
#Define the Regularization function
def L1_regularisation(model):

    flattened_weight = torch.cat([param.view(-1).to(device) for param in model.parameters()])
    
    #Calculate the norms
    l0 = abs(flattened_weight)
    l1 = abs(flattened_weight- 1)
    l_neg1 = abs(flattened_weight + 1)

    # Compute the minimum across L0, L1, and L-1
    #stacked_tensors = torch.stack([beta*l0, l1, l_neg1], dim=0)
    #min_values, _ = stacked_tensors.min(dim=0)
    min_values = l0 * l1 * l_neg1


    return min_values.mean()

In [12]:
def L2_regularisation(model):

    flattened_weight = torch.cat([param.view(-1).to(device) for param in model.parameters()])
    #Calculate the norms
    l0 = (flattened_weight)**2
    l1 = (flattened_weight -1)**2
    l_neg1 = (flattened_weight +1)**2



    # Compute the minimum across L0, L1, and L-1
    #stacked_tensors = torch.stack([beta*l0, l1, l_neg1], dim=0)
    #min_values, _ = stacked_tensors.min(dim=0)
    min_values = l0 * l1 * l_neg1
    
    return min_values.mean()

# 3. Train the network

In [13]:
import os

In [14]:
lo = []
epo = []
MSE = []
REG  = []

In [15]:
torch.save(model.state_dict(),os.path.join('2*2_initialisation.pth'))

In [15]:
#If start a training from saved model
#model.load_state_dict(torch.load('/path.pth', map_location=torch.device(device)))


/tmp/ipykernel_25/4060986437.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/kaggle/input/3-correct-initialization/3_3_initialisation 

<All keys matched successfully>

In [16]:
#Train the network
epochs = 5000
alpha = 1/50 #the hyperparameter of the regularization function

# Training loop
for epoch in range(epochs):
    total_loss = 0.0

    for inputs, inputs_, targets in train_loader:
        if torch.cuda.is_available():
            inputs = inputs.cuda()
            inputs_ = inputs_.cuda()
            targets = targets.cuda()

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs.type(torch.float32), inputs_.type(torch.float32))

        #Calculate the loss
        mse, reg = criterion(outputs, targets.type(torch.float32)), L1_regularisation(model)
        loss = mse + (alpha * reg)
        total_loss += loss.item()

        # Backward pass
        loss.backward()
        
        optimizer.step()

        # Print losses
        print(f"MSE = {mse.item()} and Regularization = {reg.item()}")
        
    #increase the pinality for the mse, so we will give it more attention for to be reduced during the training
    #if(epoch+1) % 5000 ==0:
    #  alpha = alpha *10
        
    # Print the training loss for each epoch
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss}\n")
    epo.append(epoch)
    lo.append(total_loss)
    MSE.append(mse.item())
    REG.append(reg.item())

    #save the model every 100 epoch
    if ((epoch+1) % 100 == 0):
        save_dir = '/kaggle/working/'
        torch.save(model.state_dict(),os.path.join(save_dir, f'2*2_7perceptrons_{epoch+1}epochs.pth'))

MSE = 0.23061786592006683 and Regularization = 0.24387215077877045
Epoch 1/5000, Loss: 0.23549531400203705

MSE = 0.2297186702489853 and Regularization = 0.24380162358283997
Epoch 2/5000, Loss: 0.2345947027206421

MSE = 0.22882722318172455 and Regularization = 0.24374011158943176
Epoch 3/5000, Loss: 0.23370201885700226

MSE = 0.22794367372989655 and Regularization = 0.24368076026439667
Epoch 4/5000, Loss: 0.23281729221343994

MSE = 0.22706811130046844 and Regularization = 0.24362091720104218
Epoch 5/5000, Loss: 0.23194052278995514

MSE = 0.2262006402015686 and Regularization = 0.24356001615524292
Epoch 6/5000, Loss: 0.2310718446969986

MSE = 0.2253412902355194 and Regularization = 0.24351292848587036
Epoch 7/5000, Loss: 0.2302115559577942

MSE = 0.22449012100696564 and Regularization = 0.24347323179244995
Epoch 8/5000, Loss: 0.22935958206653595

MSE = 0.2236471325159073 and Regularization = 0.24343222379684448
Epoch 9/5000, Loss: 0.22851577401161194

MSE = 0.22281236946582794 and Regul

In [25]:
#Used to load a model from a specific epoch
model.load_state_dict(torch.load('/kaggle/working/2*2_7perceptrons_2000epochs.pth', map_location=torch.device(device)))

/tmp/ipykernel_25/4079064246.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/kaggle/working/2*2_7perceptrons_2000epochs.pth', map_loca

<All keys matched successfully>

In [26]:
#Round the model's weight values to the nearest (-1,1,or 0)
for param in model.parameters():
  param.data.round_()

In [27]:
# Evaluate the model on the test set; the test is always one batch
model.eval()  
test_loss = 0.0

with torch.no_grad():
    for inputs,inputs_, targets in test_loader:
        if torch.cuda.is_available():
            inputs = inputs.cuda()
            inputs_ = inputs_.cuda()
            targets = targets.cuda()
        
        
        outputs = model(inputs.type(torch.float32),inputs_.type(torch.float32)) 
        
        #Calculate the losses separatly
        test_loss = criterion(outputs, targets.type(torch.float32)).item()
        reg = L1_regularisation(model).item()
        
        
# Calculate the average test loss
#average_test_loss = test_loss / len(test_loader)
print(f"Test Loss: {test_loss} and regularisation: {reg}")

Test Loss: 2.2166303775811693e-15 and regularisation: 0.0


In [ ]:
#Print the hyperparameters of the model (the algorithm)
for param in model.parameters():
    print(param)

In [ ]:
import matplotlib.pyplot as plt

# Plotting both the curves simultaneously
plt.figure(figsize=(10, 6))  # Width: 10 inches, Height: 6 inches

plt.plot(epo, lo, color='r', label='total loss')
plt.plot(epo, torch.tensor(MSE).cpu().numpy(), color='g', label= 'MSE')
plt.plot(epo, torch.tensor(REG).cpu().numpy(), color='b',label = 'regularization')

# Naming the x-axis, y-axis and the whole graph
plt.xlabel("Epochs")
plt.ylabel("Loss")
#plt.title("Model convergence using different regularization techniques")


# Adding legend, which helps us recognize the curve according to it's color
plt.legend()

# To load the display window
plt.show()
